# Graphene/Ni(111) Interface: Film Registry and Separation

## 0. Introduction

This notebook reproduces the registry energetics of graphene on Ni(111) following the manuscript:

> **Arjun Dahal, Matthias Batzill**
> "Graphene–nickel interfaces: a review"
> Nanoscale, 6(5), 2548. (2014)
> [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

Graphene and Ni(111) are lattice-matched to about one percent, so instead of a moiré pattern the
film locks into one registry. The manuscript's Fig. 1 shows the four it considers, and this notebook
computes all four under those names — **hollow**, **atop/fcc**, **atop/hcp** and **bridge**:

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/optimization/optimization_interface_film_xy_position_graphene_nickel/0-figure-from-manuscript.webp" alt="The four registries of graphene on a close-packed metal surface" width="600"/>

Panel **(b)**, the atop/fcc registry, is the favourable position the manuscript highlights and the
one the companion structure notebook targets.

What is reproduced:

1. **Which registry is most favourable** — total energies of the film at each registry, each at its
   own optimal separation.
2. **The chemisorption separation** — the review reports chemisorbed graphene at **0.21 nm** above
   the top Ni plane, against the **0.33 nm** van der Waals spacing of graphite. The hollow registry
   is not chemisorbed at all: it has only a dispersion-bound minimum, much further out.

The comparison runs in two tiers:

- **Fast (here, in minutes):** energy against separation for every registry with the
  [MACE-MP](https://github.com/ACEsuit/mace) machine-learned force field, including D3 dispersion.
- **Precise (platform jobs):** DFT total energy for each registry at its optimal separation, plus
  two same-cell reference jobs so an adsorption energy per carbon atom can be formed. A default run
  submits one job; activate the rest to compute the full comparison.

The atop/fcc and atop/hcp registries come out within a few meV per carbon atom of each other, which
is finer than either method here resolves, so they are treated as degenerate and the top-site family
is compared against the hollow arrangement rather than against one another.

## 1. Prepare the Environment
### 1.1. Install Packages


In [ ]:
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Set Parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "./uploads"
BASE_MATERIAL_NAME = "Graphene_Nickel_interface"  # created by the companion structure notebook

# 4. MLFF parameters
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"  # "small", "medium", "large" — large resolves the shallow chemisorbed minimum
MACE_DISPERSION = True  # D3 dispersion; the physisorbed minimum does not exist without it
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"

# 5. Separation scan: film-to-substrate plane distance, in Angstrom
Z_SCAN_START = 1.8
Z_SCAN_STOP = 4.3
Z_SCAN_STEP = 0.15

# A chemisorbing registry has two minima: one where graphene bonds to the surface and one held
# only by dispersion, further out. This splits them. Chemisorbed Gr/Ni(111) is reported near
# 2.1 A and the graphite van der Waals spacing is 3.3 A, so anything below 2.6 A is the
# chemisorbed branch by a wide margin either way.
CHEMISORBED_BELOW = 2.6  # Angstrom

# 6. Workflow parameters
WORKFLOW_SEARCH_TERM = "total_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Total Energy (Gr/Ni registry)"

# Two extra single points in the SAME cell (bare Ni slab, free-standing graphene) turn the
# interface energies into an adsorption energy per carbon atom.
COMPUTE_ADSORPTION_ENERGY = True

# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"
ECUTWFC = 50
ECUTRHO = 400  # ultrasoft Ni needs a dense charge-density grid
SCF_KGRID = [12, 12, 1]  # for the ~1x1 hexagonal interface cell; scale down for larger cells
DEGAUSS = 0.01  # Ry; the metal needs wider smearing than the template default to converge

# Nickel is ferromagnetic: run spin-polarized with a starting moment on Ni
STARTING_MAGNETIZATION = {"Ni": 0.7}
USE_VDW_D3 = True  # apply the same D3 correction in the DFT jobs (QE vdw_corr = "grimme-d3")

# 7. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 8. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30

## 2. Load the Base Interface

The base interface is created by the companion structure notebook and saved into `uploads/`.
It is required — this notebook does not substitute another material.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.modify import interface_get_part
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

base_interface = load_material_from_folder(FOLDER, BASE_MATERIAL_NAME)
if base_interface is None:
    raise RuntimeError(
        f"'{BASE_MATERIAL_NAME}' not found in {FOLDER} — run "
        "optimization_interface_film_xy_position_graphene_nickel.ipynb first."
    )

film_part = interface_get_part(base_interface, part=InterfacePartsEnum.FILM)
substrate_part = interface_get_part(base_interface, part=InterfacePartsEnum.SUBSTRATE)

_cart = base_interface.clone()
_cart.to_cartesian()
film_cart = film_part.clone(); film_cart.to_cartesian()
substrate_cart = substrate_part.clone(); substrate_cart.to_cartesian()

film_z = [c[2] for c in film_cart.basis.coordinates.values]
substrate_z = [c[2] for c in substrate_cart.basis.coordinates.values]
measured_gap = min(film_z) - max(substrate_z)

print(f"Material:  {base_interface.name}")
from collections import Counter
composition = dict(Counter(base_interface.basis.elements.values))
print(f"Composition: {composition}")
print(f"Atoms:     {len(base_interface.basis.elements.values)} "
      f"({len(film_cart.basis.elements.values)} film C, {len(substrate_cart.basis.elements.values)} substrate Ni)")
print(f"Film-substrate plane distance as built: {measured_gap:.3f} A")

visualize([{"material": base_interface, "title": base_interface.name}], repetitions=[3, 3, 1], rotation="-90x")

## 3. Place the Film at the High-Symmetry Registries

The registries are defined by where carbon atoms sit relative to the Ni(111) surface sites:
**top** (above a first-layer Ni), **hcp hollow** (above a second-layer Ni), **fcc hollow**
(above a third-layer Ni), and **bridge** (midpoint of two neighboring first-layer Ni).
The sites are measured from the structure itself — the top three Ni layers — and the film is
translated so one carbon sublattice lands on each site in turn.


In [ ]:
import numpy as np

cell_2d = np.array(_cart.lattice.vector_arrays)[:2, :2]

ni_xyz = np.array(substrate_cart.basis.coordinates.values)
z_values = sorted(set(np.round(ni_xyz[:, 2], 2)), reverse=True)
layer_tol = 0.5
layers = []
for z in z_values:
    if layers and abs(z - layers[-1][0]) < layer_tol:
        continue
    layers.append((z, ni_xyz[np.abs(ni_xyz[:, 2] - z) < layer_tol]))
if len(layers) < 3:
    raise RuntimeError(f"Need >= 3 Ni layers to locate the fcc and hcp sites, found {len(layers)}")

c_xyz = np.array(film_cart.basis.coordinates.values)
if len(c_xyz) != 2:
    raise RuntimeError(f"Expected a 1x1 graphene film (2 carbons), found {len(c_xyz)}")
c_a, c_b = c_xyz[0], c_xyz[1]

def nearest_image(site_xy, point_xy):
    """The periodic image of site_xy closest to point_xy."""
    images = [site_xy + i * cell_2d[0] + j * cell_2d[1] for i in (-1, 0, 1) for j in (-1, 0, 1)]
    return min(images, key=lambda s: np.linalg.norm(s - point_xy))

# Surface sites read off the structure itself: a first-layer Ni marks an atop site, a second-layer
# Ni projects onto the hcp hollow and a third-layer Ni onto the fcc hollow.
site_xy = {
    "atop": nearest_image(layers[0][1][0][:2], c_a[:2]),
    "hcp": nearest_image(layers[1][1][0][:2], c_a[:2]),
    "fcc": nearest_image(layers[2][1][0][:2], c_a[:2]),
}
shortest_lattice_vector = min((cell_2d[0], cell_2d[1], cell_2d[0] + cell_2d[1], cell_2d[0] - cell_2d[1]),
                              key=np.linalg.norm)

def site_of(point_xy):
    """Which named site a carbon lands on. Refuses to guess when two are equidistant."""
    distances = {name: np.linalg.norm(nearest_image(site, point_xy) - point_xy)
                 for name, site in site_xy.items()}
    ordered = sorted(distances.items(), key=lambda kv: kv[1])
    if len(ordered) > 1 and abs(ordered[0][1] - ordered[1][1]) < 0.05:
        return None
    return ordered[0][0]

# The manuscript's Fig. 1: (a) hollow, (b) atop/fcc, (c) atop/hcp, (d) bridge. In a 1x1 cell the two
# carbon sublattices sit on two of the three named sites, which gives the first three; the bridge
# registry is defined by its own geometry — one carbon on the midpoint between neighbouring
# first-layer Ni — and no site label is claimed for the other.
displacements = {"bridge": np.array([*(site_xy["atop"] + shortest_lattice_vector / 2 - c_a[:2]), 0.0])}
for a_site in ("fcc", "atop", "hcp"):
    shift = np.array([*(site_xy[a_site] - c_a[:2]), 0.0])
    b_site = site_of(c_b[:2] + shift[:2])
    if b_site is None:
        raise RuntimeError(f"Carbon B is equidistant from two sites for the {a_site} placement")
    pair = {a_site, b_site}
    label = f"atop_{(pair - {'atop'}).pop()}" if "atop" in pair else "hollow"
    displacements[label] = shift

expected = {"hollow", "atop_fcc", "atop_hcp", "bridge"}
if set(displacements) != expected:
    raise RuntimeError(f"Registry derivation produced {set(displacements)}, expected {expected}")

print(f"{'registry':<12}{'manuscript Fig. 1':<22}{'film shift (A)'}")
for label, panel in (("hollow", "(a) hollow site"), ("atop_fcc", "(b) atop/'fcc' site"),
                     ("atop_hcp", "(c) atop/'hcp' site"), ("bridge", "(d) bridge site")):
    print(f"{label:<12}{panel:<22}{np.round(displacements[label][:2], 3)}")


In [ ]:
from mat3ra.made.tools.modify import interface_displace_part

def film_at(registry_label, plane_distance):
    displacement = displacements[registry_label] + np.array([0.0, 0.0, plane_distance - measured_gap])
    return interface_displace_part(base_interface, displacement=list(displacement))

preview = []
for label in displacements:
    m = film_at(label, measured_gap)
    m.name = f"{BASE_MATERIAL_NAME} {label}"
    preview.append({"material": m, "title": label})

visualize(preview, repetitions=[2, 2, 1])

## 4. Energy vs. Separation with MACE

For each registry the film is rigidly moved through a range of plane distances and the energy is
computed with MACE-MP + D3. The energy curve of a chemisorbing registry has **two minima** — a
chemisorbed one near 2 A and a dispersion-bound one near the van der Waals distance — while the
hollow registry only has the dispersion-bound minimum. The registry comparison therefore reads the
**chemisorbed branch**: each chemisorbing registry is compared at its own chemisorbed minimum, and
a registry with no such minimum is reported as non-chemisorbing, which is the manuscript's own
statement about the hollow arrangement.


In [ ]:
from mat3ra.made.tools.convert import to_ase
from mat3ra.notebooks_utils.mlff import create_mlff_calculator

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": MACE_DISPERSION,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)

In [ ]:
distances = np.arange(Z_SCAN_START, Z_SCAN_STOP + 1e-9, Z_SCAN_STEP)
n_carbon = len(film_cart.basis.elements.values)

def refine_minimum(x, y, i):
    if 0 < i < len(x) - 1:
        coefficients = np.polyfit(x[i - 1:i + 2], y[i - 1:i + 2], 2)
        d = float(-coefficients[1] / (2 * coefficients[0]))
        return d, float(np.polyval(coefficients, d))
    return float(x[i]), float(y[i])

scan_results = {}
for label in displacements:
    energies = []
    for d in distances:
        atoms = to_ase(film_at(label, float(d)))
        atoms.calc = calculator
        energies.append(float(atoms.get_potential_energy()))
    energies = np.array(energies)
    # interior minima only: a point at the scan edge is not a minimum
    minima = [refine_minimum(distances, energies, i)
              for i in range(1, len(energies) - 1)
              if energies[i] < energies[i - 1] and energies[i] < energies[i + 1]]
    chem = min((m for m in minima if m[0] < CHEMISORBED_BELOW), key=lambda m: m[1], default=None)
    phys = min((m for m in minima if m[0] >= CHEMISORBED_BELOW), key=lambda m: m[1], default=None)
    lowest_sampled = float(distances[int(np.argmin(energies))])
    if chem is not None and lowest_sampled <= distances[1]:
        print(f"! {label}: minimum sits at the low edge of the scan ({lowest_sampled:.2f} A) — "
              f"lower Z_SCAN_START before trusting it")
    scan_results[label] = {"distances": distances, "energies": energies, "chem": chem, "phys": phys}
    chem_text = f"chemisorbed at {chem[0]:.2f} A" if chem else "does not chemisorb"
    phys_text = f"physisorbed at {phys[0]:.2f} A" if phys else "no physisorbed minimum in range"
    print(f"{label:<16} {chem_text:<28} {phys_text}")

In [ ]:
import plotly.graph_objects as go

reference = min(min(m[1] for m in (r["chem"], r["phys"]) if m) for r in scan_results.values())
fig = go.Figure()
for label, r in scan_results.items():
    fig.add_trace(go.Scatter(x=r["distances"], y=(r["energies"] - reference) * 1000 / n_carbon,
                             mode="lines+markers", name=label))
fig.update_layout(
    title="Energy vs. film-substrate distance (MACE-MP + D3)",
    xaxis_title="plane distance (A)",
    yaxis_title="energy relative to the deepest minimum (meV / C atom)",
    yaxis_range=[-20, 300],
)
fig.show()


In [ ]:
chemisorbing = {label: r for label, r in scan_results.items() if r["chem"] is not None}
if not chemisorbing:
    raise RuntimeError("No registry shows a chemisorbed minimum — check the MACE model settings")
ranked = sorted(chemisorbing.items(), key=lambda kv: kv[1]["chem"][1])
winner = ranked[0][0]
e_winner = ranked[0][1]["chem"][1]

print("Chemisorbed branch (the registry comparison):")
print(f"{'registry':<16}{'d_chem (A)':<12}{'dE (meV/C)':<12}")
for label, r in ranked:
    print(f"{label:<16}{r['chem'][0]:<12.2f}{(r['chem'][1] - e_winner) * 1000 / n_carbon:<12.1f}")
for label, r in scan_results.items():
    if r["chem"] is None:
        where = f"minimum at {r['phys'][0]:.2f} A" if r["phys"] else "no minimum in range"
        print(f"{label:<16}does not chemisorb — {where}")

# The two atop registries differ by a few meV per carbon, which is finer than a machine-learned
# force field resolves; treat them as degenerate and compare the atop family against the hollow.
gap_to_runner_up = ((ranked[1][1]["chem"][1] - e_winner) * 1000 / n_carbon) if len(ranked) > 1 else None
print(f"\nLowest chemisorbed registry: {winner}"
      + (f" (next is {ranked[1][0]}, +{gap_to_runner_up:.1f} meV/C)" if gap_to_runner_up is not None else ""))


## 5. Total Energy with DFT on the Platform

The MACE scan is the fast survey; the platform computes DFT total energies for the registries,
each at its own optimal separation. A default run submits **one** job. To compute the full
comparison and the final verdict, uncomment the remaining registries below and re-run from here.


In [ ]:
DFT_REGISTRY_NAMES = [
    "atop_fcc",
    # "atop_hcp",
    # "bridge",
    # "hollow",
]


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({project_id})")

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

def submitted_copy(material, name):
    """QE needs ATOMIC_SPECIES and ATOMIC_POSITIONS to agree, and the film/substrate labels only
    served the displacement, so they are dropped from anything submitted."""
    m = material.clone()
    m.basis.labels.values = []
    m.name = name
    return Material.create(get_or_create_material(client, m, ACCOUNT_ID))

dft_materials = {}
for label in DFT_REGISTRY_NAMES:
    branch = scan_results[label]["chem"] or scan_results[label]["phys"]
    if branch is None:
        raise RuntimeError(f"{label} has no minimum in the scan window — widen the scan before submitting")
    d_eq = branch[0]
    saved = submitted_copy(film_at(label, d_eq), f"{BASE_MATERIAL_NAME} {label} d{d_eq:.2f}")
    dft_materials[label] = saved
    print(f"{label:<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms, d = {d_eq:.2f} A)")

# The references live in the SAME cell as the interface, so the cell, k-grid, cutoffs and smearing
# cancel out of the difference and what remains is the adsorption energy.
reference_materials = {}
if COMPUTE_ADSORPTION_ENERGY:
    for name, part in (("substrate", substrate_part), ("film", film_part)):
        saved = submitted_copy(part, f"{BASE_MATERIAL_NAME} {name} reference")
        reference_materials[name] = saved
        print(f"{name + ' ref':<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")


In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft",
    subtype="gga",
    functional=FUNCTIONAL,
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

In [ ]:
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input

reference_material = dft_materials[DFT_REGISTRY_NAMES[0]]
scf_subworkflow = workflow.subworkflows[0]

unit = scf_subworkflow.get_unit_by_name(name="pw_scf")
unit.add_context(PointsGridDataProvider(material=reference_material, dimensions=SCF_KGRID,
                                        isEdited=True).get_context_item_data())
unit.add_context(PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO,
                                                 isEdited=True).get_context_item_data())
scf_subworkflow.set_unit(unit)

# ATOMIC_SPECIES is ordered by first appearance of each element
species_names = []
for element in reference_material.basis.elements.values:
    if element not in species_names:
        species_names.append(element)

system_patch = {"nspin": 2, "degauss": DEGAUSS}
for atomic_species, value in STARTING_MAGNETIZATION.items():
    matches = [i for i, name in enumerate(species_names) if name.startswith(atomic_species)]
    for index in matches:
        system_patch[f"starting_magnetization({index + 1})"] = value
if USE_VDW_D3:
    system_patch["vdw_corr"] = "grimme-d3"

patch_workflow_qe_input(workflow, {"system": system_patch}, unit_names=["pw_scf"])
print(f"ATOMIC_SPECIES order: {species_names}")
print(f"&SYSTEM patch: {system_patch}")

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

In [ ]:
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.job import create_job

def submit_job_for(label, saved_material):
    job_response = create_job(
        api_client=client,
        materials=[saved_material],
        workflow=workflow,
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {label} {timestamp}",
        compute=compute.to_dict(),
    )
    job_id = dict_to_namespace_recursive(job_response)._id
    print(f"{label:<16} -> job {job_id}")
    return job_id

jobs = {label: submit_job_for(label, m) for label, m in dft_materials.items()}
reference_jobs = {name: submit_job_for(f"{name} reference", m) for name, m in reference_materials.items()}


In [ ]:
for label, job_id in {**jobs, **reference_jobs}.items():
    client.jobs.submit(job_id)
    print(f"Submitted {label}: {job_id}")


In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

all_job_ids = list(jobs.values()) + list(reference_jobs.values())
if not all_job_ids:
    raise RuntimeError("No jobs were created — nothing to wait for.")
await wait_for_jobs_to_finish_async(client.jobs, all_job_ids, poll_interval=POLL_INTERVAL)


In [ ]:
from mat3ra.prode import PropertyName

def total_energy_of(job_id):
    property_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
    return float(property_data[0]["data"]["value"])

dft_energies = {label: total_energy_of(job_id) for label, job_id in jobs.items()}
reference_energies = {name: total_energy_of(job_id) for name, job_id in reference_jobs.items()}

dft_winner = min(dft_energies, key=dft_energies.get)
print(f"{'registry':<16}{'E_DFT (eV)':<16}{'dE (meV/C)':<12}{'d (A)'}")
for label, e in sorted(dft_energies.items(), key=lambda kv: kv[1]):
    de = (e - dft_energies[dft_winner]) * 1000 / n_carbon
    print(f"{label:<16}{e:<16.4f}{de:<12.1f}{(scan_results[label]['chem'] or scan_results[label]['phys'])[0]:.2f}")

adsorption_energies = {}
if len(reference_energies) == 2:
    separated = reference_energies["substrate"] + reference_energies["film"]
    adsorption_energies = {label: (e - separated) / n_carbon for label, e in dft_energies.items()}


## 6. Compare with the Article


In [ ]:
# What the review states: chemisorbed graphene sits 0.21 nm above Ni(111), against the 0.33 nm
# van der Waals spacing of graphite, and its Fig. 1b — the atop/fcc registry — is the favourable
# position. The atop/fcc and atop/hcp registries differ by a few meV per carbon here, below what
# this method resolves, so the check is on the atop family rather than on one of the two.
PAPER_CHEMISORBED_DISTANCE = 2.1  # A, from 0.21 nm
PAPER_VDW_DISTANCE = 3.3  # A, from 0.33 nm — graphite reference, reported for context
TOLERANCE_CHEMISORBED = 0.15  # A

dft_energies = globals().get("dft_energies", {})
hollow = scan_results["hollow"]
hollow_branch = hollow["chem"] or hollow["phys"]
hollow_text = f"{hollow_branch[0]:.2f} A" if hollow_branch else "none in the scan window"

checks = {
    "an atop registry is the most favourable": winner.startswith("atop_"),
    "it chemisorbs at the reported distance": abs(scan_results[winner]["chem"][0] - PAPER_CHEMISORBED_DISTANCE) <= TOLERANCE_CHEMISORBED,
    "the hollow registry does not chemisorb": hollow["chem"] is None,
}

print(f"most favourable registry   {winner:<16} review: atop/fcc (Fig. 1b)")
print(f"its separation             {scan_results[winner]['chem'][0]:.2f} A            review: {PAPER_CHEMISORBED_DISTANCE} A (0.21 nm)")
print(f"hollow registry minimum    {hollow_text:<16} review: beyond the vdW gap ({PAPER_VDW_DISTANCE} A in graphite)")
for name, ok in checks.items():
    print(f"  {'ok  ' if ok else 'FAIL'} {name}")
print(f"\nReproduces Dahal & Batzill (2014) [MACE tier]: {'yes' if all(checks.values()) else 'no'}")

adsorption = globals().get("adsorption_energies", {})
if adsorption:
    print()
    for label, e_ads in sorted(adsorption.items(), key=lambda kv: kv[1]):
        print(f"adsorption energy {label:<12} {e_ads * 1000:7.1f} meV per C atom")
    print("(PBE+D3 in this cell; the review collates values from several methods, so compare the "
          "ordering and the magnitude, not the digits)")

if len(dft_energies) == len(displacements):
    dft_ranked = sorted(dft_energies.items(), key=lambda kv: kv[1])
    dft_ok = dft_ranked[0][0].startswith("atop_")
    print(f"most favourable registry   {dft_ranked[0][0]:<16} review: atop/fcc (Fig. 1b)   [DFT]")
    print(f"Reproduces Dahal & Batzill (2014) [DFT tier]: {'yes' if dft_ok else 'no'}")
else:
    remaining = [l for l in displacements if l not in dft_energies]
    print(f"DFT ran for {len(dft_energies)} of {len(displacements)} registries — add {remaining} "
          f"to DFT_REGISTRY_NAMES for the DFT-tier verdict.")


## References

[1] Arjun Dahal, Matthias Batzill, "Graphene-nickel interfaces: a review",
Nanoscale 6(5), 2548 (2014). [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

[2] mat3ra-made: https://github.com/Exabyte-io/made

[3] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace
